# La régression par cohorte

La même régression que dans `Regression_Sans_Anomalies` : les mêmes six modèles, un train (80 %), un test (20 %), les mêmes métriques à la fin.

La seule différence : au lieu d'un seul modèle pour toute la banque, **chaque cohorte a son modèle**.

Pour une cohorte, ça se passe comme ça :

1. on prend les salariés de la cohorte ;
2. ceux qui sont dans le train servent à apprendre le modèle ;
3. le modèle prédit ceux qui sont dans le test.

On fait pareil pour toutes les cohortes, on rassemble les prédictions, et on calcule les métriques.

Il y a plusieurs façons de former les cohortes (avec l'ancienneté, le diplôme, la catégorie d'école…). Le notebook les essaie toutes, donne les métriques de chacune, puis les classe de la meilleure à la moins bonne.

La ligne **Sans cohorte** des tableaux est le repère : on ne découpe rien, un seul modèle apprend sur tout le train.

---
## 1. Les salariés

Avec `ECARTER_LES_ANOMALIES = False`, personne n'est écarté. Avec `True`, la détection du projet tourne et les salariés des priorités de la liste `A_RETIRER` sortent avant la régression.

In [ ]:
import sys

import numpy as np
import pandas as pd
import yaml
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

# Les fonctions de la détection sont celles du projet, dans src/
sys.path.append('../src')
from anomaly_pretraitement import preparer_employes
from anomaly_cohortes import former_cohortes
from anomaly_regles import apply_rulebook
from anomaly_signal_iforest import ml_anomaly
from anomaly_signal_fort import apply_ml_strong_signal
from anomaly_score_general import aggregate_risk
from anomaly_recommandations import recommendations

DOSSIER = '../input'   # le dossier des trois fichiers Excel
pd.set_option('display.precision', 2)   # 2 chiffres après la virgule à l'affichage
pd.set_option('display.max_colwidth', None)   # les textes des tableaux s'affichent en entier

with open('../config/rules.yaml', encoding='utf-8') as fichier:
    regles = yaml.safe_load(fichier)['rules']

# ECARTER_LES_ANOMALIES. False : personne n'est écarté. True : les salariés des priorités de A_RETIRER sortent.
ECARTER_LES_ANOMALIES = False
A_RETIRER = ['Critical']   # ne sert que si ECARTER_LES_ANOMALIES = True. Pour en retirer plus : ['Critical', 'Major']

df = preparer_employes(f'{DOSSIER}/employes.xlsx', f'{DOSSIER}/bands.xlsx', f'{DOSSIER}/market.xlsx')
print(f"{len(df)} salariés.")

if ECARTER_LES_ANOMALIES:
    # La détection du projet, pour repérer les anomalies
    detection = df
    for etape in (former_cohortes, apply_rulebook, ml_anomaly, apply_ml_strong_signal, aggregate_risk, recommendations):
        detection = etape(detection, regles)
    df = df[~detection['Severity'].isin(A_RETIRER)]
    print(f"{len(df)} salariés une fois les anomalies écartées.")

# Ce que Regression_Sans_Anomalies calcule en plus, avec les mêmes noms
df['Compa_Ratio'] = df['CompaRatio']
df['Age_Embauche'] = df['Age'] - df['Anciennete']
# Une division par zéro donne « infini » : on remet la case à vide
df = df.replace([np.inf, -np.inf], np.nan)
# Un ratio négatif ou nul est impossible (salaire à 0 ou négatif) : case vidée aussi
for c in ['Compa_Ratio', 'Ratio_Grille_Marche']:
    df.loc[df[c] <= 0, c] = np.nan
# Un salarié sans Compa-Ratio (pas de grille, ou pas de salaire) ne peut pas servir : on le retire
df = df[df['Compa_Ratio'].notna()].reset_index(drop=True)
print(f"{len(df)} salariés avec une grille et un salaire.")

---
## 2. Les cohortes à essayer

C'est ici qu'on règle les cohortes. Dans `COHORTES`, **une ligne = une façon de former les cohortes** :

* à gauche, le nom qu'on lui donne (il sert dans les tableaux) ;
* à droite, entre crochets, les colonnes qui forment la cohorte.

Une cohorte, ce sont tous les salariés qui ont les mêmes valeurs dans ces colonnes. Avec `['Niveau', 'Job_Family']`, les salariés de niveau 6 en Retail Banking forment une cohorte, ceux de niveau 5 en Finance une autre. Chaque salarié est dans une seule cohorte.

Une cohorte de moins de `TAILLE_MIN` salariés est trop petite pour apprendre un modèle : ses salariés vont dans une cohorte à part, « Autres ».

Pour ajouter une façon de faire : recopier une ligne et changer les colonnes. Un `#` devant une ligne la met de côté.

Les colonnes qu'on peut utiliser :

| Colonne | Ce que c'est |
|---|---|
| `Niveau` | le niveau du poste |
| `Job_Family` | le métier |
| `Anciennete_Bucket` | la tranche d'ancienneté (0-2, 3-5, 6-12, 13-20, plus de 20 ans) |
| `Niveau_Etude_Max` | le niveau d'études (BAC, BAC+2, BAC+3…) |
| `Tranche_Etude` | le niveau d'études en trois tranches (BAC+2 ou moins, BAC+3 ou BAC+4, BAC+5 et plus) |
| `Cat_Ecole_Max` | la catégorie d'école (ingénieur, commerce, université, autres) |
| `Entite_N1` | le pôle |
| `Performance_N1`, `Positionnement_9BOX`, `Hot_job` | la performance, la case 9-Box, la tension du métier |

Une case vide (une catégorie d'école inconnue, par exemple) s'écrit « nan » dans le nom de la cohorte : les salariés concernés sont ensemble.

In [ ]:
# La taille minimale d'une cohorte : celle de la détection (15). Mettre un autre nombre ici pour la changer.
TAILLE_MIN = regles['cohort_min_size']

COHORTES = {
    # La cohorte de la détection
    'Niveau + métier + ancienneté':                   ['Niveau', 'Job_Family', 'Anciennete_Bucket'],
    # Avec le niveau d'études et la catégorie d'école
    'Niveau + métier + diplôme':                      ['Niveau', 'Job_Family', 'Niveau_Etude_Max'],
    'Niveau + métier + école':                        ['Niveau', 'Job_Family', 'Cat_Ecole_Max'],
    'Niveau + métier + diplôme + école':              ['Niveau', 'Job_Family', 'Niveau_Etude_Max', 'Cat_Ecole_Max'],
    'Niveau + métier + ancienneté + diplôme':         ['Niveau', 'Job_Family', 'Anciennete_Bucket', 'Niveau_Etude_Max'],
    'Niveau + métier + ancienneté + école':           ['Niveau', 'Job_Family', 'Anciennete_Bucket', 'Cat_Ecole_Max'],
    'Niveau + métier + ancienneté + diplôme + école': ['Niveau', 'Job_Family', 'Anciennete_Bucket', 'Niveau_Etude_Max', 'Cat_Ecole_Max'],
    # D'autres façons de faire
    'Niveau + métier + tranche de diplôme':           ['Niveau', 'Job_Family', 'Tranche_Etude'],
    'Niveau + métier':                                ['Niveau', 'Job_Family'],
    'Niveau seul':                                    ['Niveau'],
    'Métier seul':                                    ['Job_Family'],
}
AUTRES = 'Autres (cohortes trop petites)'

# Pour chaque façon de faire : la cohorte de chaque salarié
cohorte = pd.DataFrame(index=df.index)
for nom, colonnes in COHORTES.items():
    # Le nom de la cohorte : les valeurs du salarié dans ces colonnes, collées avec des « | » (« 6 | RETAIL BANKING | 0-2 »).
    # fillna('nan') : une case vide s'écrit « nan », quelle que soit la version de pandas.
    nom_cohorte = df[colonnes].astype(str).fillna('nan').agg(' | '.join, axis=1)
    # Une cohorte de moins de TAILLE_MIN salariés est trop petite : ses salariés vont dans « Autres »
    trop_petite = nom_cohorte.map(nom_cohorte.value_counts()) < TAILLE_MIN
    cohorte[nom] = nom_cohorte.mask(trop_petite, AUTRES)


---
## 3. Les colonnes du modèle

La liste de `Regression_Sans_Anomalies`. Un `#` devant une colonne la met de côté.

Une colonne qui a servi à former la cohorte est la même pour tous dans la cohorte : elle n'explique plus rien à l'intérieur. Elle reste dans la liste pour le modèle « Sans cohorte » et pour la cohorte « Autres ».

In [ ]:
CIBLES = ['Fixe_Annuel_MAD', 'Compa_Ratio']   # les deux variables à prédire : le salaire fixe et le Compa-Ratio

COLONNES = [
    'Niveau',                  # le niveau du poste
    'Niveau_Etude_Max',        # le niveau d'études
    #'Cat_Ecole_Max',           # la catégorie d'école
    'Age_Embauche',            # âge − ancienneté
    'Anciennete',              # l'ancienneté, en années
    'Competence_N1',           # la note de compétence de l'an dernier
    'Performance_N1',          # la performance de l'an dernier
    'Hot_job',                 # la tension du métier sur le marché
    'Ratio_Grille_Marche',     # Mid / médiane du marché
]
donnees = df[CIBLES + [c for c in COLONNES if c in df.columns]].copy()

# 1. La performance et le niveau d'études passent en rang. Un libellé inconnu prend le rang médian.
RANG_PERFORMANCE = {"Très en dessous": 1, "Proche de l'objectif": 2, "A l'objectif": 3, "Supérieur à l'objectif": 4, "Très au-delà": 5}
RANG_ETUDE = {'SECONDAIRE': 0, 'BAC': 1, 'BAC+1': 2, 'BAC+2': 3, 'BAC+3': 4, 'BAC+4': 5, 'BAC+5': 6, 'BAC+6': 6,
              'BAC+7': 7, 'EXPERTISE COMPTABLE': 7}
for colonne, rangs in [('Performance_N1', RANG_PERFORMANCE), ('Niveau_Etude_Max', RANG_ETUDE)]:
    if colonne in donnees.columns and donnees[colonne].isin(list(rangs)).mean() > 0.5:
        donnees[colonne] = donnees[colonne].map(rangs)
        donnees[colonne] = donnees[colonne].fillna(donnees[colonne].median())

# 2. Les colonnes restées en texte ne sont pas utilisées : les modèles ne lisent que des chiffres
en_texte = list(donnees.select_dtypes(exclude='number').columns)
if en_texte:
    print(f"Colonnes en texte non utilisées : {en_texte}")
    donnees = donnees.drop(columns=en_texte)

# 3. Un salarié qui a une case vide dans une des colonnes choisies sort : le modèle ne sait pas lire une case vide
vides = donnees.isna().any(axis=1)
donnees = donnees[~vides]
print(f"{len(donnees)} salariés pour la suite ({int(vides.sum())} retirés pour une case vide dans une des colonnes choisies).")

# 4. Les colonnes que les modèles reçoivent : celles de la liste, sans les deux cibles
X = donnees.drop(columns=CIBLES)
print(f"{X.shape[1]} colonnes : {list(X.columns)}")

---
## 4. Le train et le test

On coupe une seule fois, comme dans `Regression_Sans_Anomalies` : 80 % des salariés dans le train, pour apprendre, et 20 % dans le test, pour vérifier.

Chaque cohorte a donc des salariés dans le train et des salariés dans le test. Le tableau donne, pour chaque ligne de `COHORTES` : le nombre de cohortes, la taille (le nombre de salariés), le train et le test.

Pourquoi couper une seule fois, et pas cohorte par cohorte : toutes les façons de former les cohortes sont ainsi jugées sur les mêmes salariés du test. On peut les comparer.

In [ ]:
# 1. On coupe en train et test. random_state : le même découpage à chaque lancement.
train, test = train_test_split(X.index, test_size=0.2, random_state=42)

# 2. Une cohorte qui n'a personne dans le train ne peut pas apprendre : ses salariés sortent du test,
#    pour toutes les façons de faire (elles restent jugées sur les mêmes salariés)
sans_train = pd.DataFrame({nom: ~cohorte.loc[test, nom].isin(cohorte.loc[train, nom]) for nom in COHORTES}).any(axis=1)
if sans_train.any():
    test = test[~sans_train.to_numpy()]
    print(f"{int(sans_train.sum())} salariés sortis du test : leur cohorte n'a personne dans le train.")
print(f"Train : {len(train)} salariés. Test : {len(test)} salariés.")

# 3. Le tableau des cohortes : une ligne par façon de former les cohortes
tableau_cohortes = pd.DataFrame({
    'Cohortes': {nom: ' + '.join(colonnes) for nom, colonnes in COHORTES.items()},
    'Nombre de cohortes': {nom: cohorte.loc[X.index, nom].nunique() for nom in COHORTES},
    'Taille': len(X),      # les salariés
    'Train': len(train),   # ceux qui servent à apprendre
    'Test': len(test),     # ceux qui servent à vérifier
}).set_index('Cohortes')
display(tableau_cohortes)

---
## 5. Les six modèles, par cohorte

Les mêmes six modèles que dans `Regression_Sans_Anomalies`, avec les mêmes réglages : M1, M2 et M5 prédisent le salaire, M3, M4 et M6 le Compa-Ratio ; une régression linéaire, un arbre de décision, un gradient boosting.

Pour chaque façon de former les cohortes : dans chaque cohorte, le modèle apprend sur les salariés de la cohorte qui sont dans le train, et prédit ceux qui sont dans le test.

La dernière ligne des tableaux, **Sans cohorte**, est le repère : on ne découpe rien, un seul modèle apprend une fois sur tout le train.

Cette cellule est la plus longue : elle refait les six modèles pour chaque ligne de `COHORTES`. Une ligne s'affiche à chaque façon de faire terminée. Pour aller plus vite, mettre un `#` devant des lignes de `COHORTES` ou de `MODELES` (les deux gradient boosting, M5 et M6, prennent presque tout le temps).

In [ ]:
# 1. Les six modèles et leurs réglages, comme dans Regression_Sans_Anomalies.
#    PROFONDEUR : le nombre de questions que l'arbre peut enchaîner.
PROFONDEUR = 5
# LOG_SALAIRE. True : M1, M2 et M5 apprennent le logarithme du salaire. False : le salaire tel quel.
LOG_SALAIRE = True
# LOG_COMPA_RATIO. True : M3, M4 et M6 apprennent le logarithme du Compa-Ratio. False : le Compa-Ratio tel quel.
LOG_COMPA_RATIO = True
salaire = 'le salaire (en log)' if LOG_SALAIRE else 'le salaire'
compa_ratio = 'le Compa-Ratio (en log)' if LOG_COMPA_RATIO else 'le Compa-Ratio'
MODELES = {
    'M1': (f'{salaire}, par une régression', 'Fixe_Annuel_MAD', LinearRegression()),
    'M2': (f'{salaire}, par un arbre', 'Fixe_Annuel_MAD', DecisionTreeRegressor(max_depth=PROFONDEUR, random_state=42)),
    'M3': (f'{compa_ratio}, par une régression', 'Compa_Ratio', LinearRegression()),
    'M4': (f'{compa_ratio}, par un arbre', 'Compa_Ratio', DecisionTreeRegressor(max_depth=PROFONDEUR, random_state=42)),
    'M5': (f'{salaire}, par un gradient boosting', 'Fixe_Annuel_MAD', HistGradientBoostingRegressor(random_state=42)),
    'M6': (f'{compa_ratio}, par un gradient boosting', 'Compa_Ratio', HistGradientBoostingRegressor(random_state=42)),
}
# Les modèles du salaire reçoivent en plus le Mid : sans lui, ils ne connaissent pas la grille du poste.
EN_PLUS_POUR_LE_SALAIRE = ['Mid']


def salaire_predit(modele, cible, apprend, verifie):
    # Le modèle apprend sur les salariés « apprend », puis prédit le salaire (en MAD) des salariés « verifie ».
    # clone : un modèle neuf à chaque fois, qui n'a rien gardé de l'apprentissage d'avant.
    modele = clone(modele)
    if cible == 'Fixe_Annuel_MAD':
        en_plus = [c for c in EN_PLUS_POUR_LE_SALAIRE if c in df.columns and c not in X.columns]
        if LOG_SALAIRE:
            X_modele = X.join(np.log(df[en_plus]).add_prefix('Log_'))
            modele.fit(X_modele.loc[apprend], np.log(df.loc[apprend, cible]))
            return np.exp(modele.predict(X_modele.loc[verifie]))
        X_modele = X.join(df[en_plus])
        modele.fit(X_modele.loc[apprend], df.loc[apprend, cible])
        return modele.predict(X_modele.loc[verifie])
    # Les modèles du Compa-Ratio : salaire prédit = Compa-Ratio prédit × Mid
    if LOG_COMPA_RATIO:
        modele.fit(X.loc[apprend], np.log(df.loc[apprend, cible]))
        return np.exp(modele.predict(X.loc[verifie])) * df.loc[verifie, 'Mid']
    modele.fit(X.loc[apprend], df.loc[apprend, cible])
    return modele.predict(X.loc[verifie]) * df.loc[verifie, 'Mid']


def salaire_predit_par_cohorte(modele, cible, nom):
    # Un modèle par cohorte : il apprend sur les salariés de la cohorte qui sont dans le train,
    # puis prédit les salariés de la cohorte qui sont dans le test
    y_pred = pd.Series(index=test, dtype=float)
    for c in cohorte.loc[test, nom].unique():
        apprend = train[(cohorte.loc[train, nom] == c).to_numpy()]
        verifie = test[(cohorte.loc[test, nom] == c).to_numpy()]
        y_pred.loc[verifie] = np.asarray(salaire_predit(modele, cible, apprend, verifie))
    return y_pred


def metriques(cible, y_pred):
    # MAE et RMSE sur le salaire, en MAD. R² sur ce que le modèle prédit : le salaire, ou le Compa-Ratio.
    y_test, mid = df.loc[test, 'Fixe_Annuel_MAD'], df.loc[test, 'Mid']
    return {'MAE (MAD)': mean_absolute_error(y_test, y_pred), 'RMSE (MAD)': mean_squared_error(y_test, y_pred) ** 0.5,
            'R²': r2_score(y_test, y_pred) if cible == 'Fixe_Annuel_MAD' else r2_score(y_test / mid, y_pred / mid)}


# 2. Pour chaque façon de former les cohortes, chaque modèle apprend par cohorte et prédit les salariés du test
REPERE = 'Sans cohorte'
resultats = []
for nom in COHORTES:
    for modele_nom, (quoi, cible, modele) in MODELES.items():
        resultats.append({'Cohorte': nom, 'Modèle': modele_nom, **metriques(cible, salaire_predit_par_cohorte(modele, cible, nom))})
    print(f"Fait : {nom}")
# Le repère : les mêmes modèles appris une seule fois sur tout le train
for modele_nom, (quoi, cible, modele) in MODELES.items():
    resultats.append({'Cohorte': REPERE, 'Modèle': modele_nom, **metriques(cible, np.asarray(salaire_predit(modele, cible, train, test)))})
resultats = pd.DataFrame(resultats)

# 3. Un tableau par métrique : une ligne par façon de former les cohortes, une colonne par modèle
display(pd.Series({modele_nom: quoi for modele_nom, (quoi, cible, modele) in MODELES.items()}).to_frame('Ce que fait le modèle'))
for metrique in ['MAE (MAD)', 'RMSE (MAD)', 'R²']:
    tableau = resultats.pivot(index='Cohorte', columns='Modèle', values=metrique).loc[list(COHORTES) + [REPERE]]
    display(tableau.round(2).style.format('{:.2f}').set_caption(metrique))

# 4. La plus petite MAE de tous les essais
meilleur = resultats.loc[resultats['MAE (MAD)'].idxmin()]
print(f"La plus petite MAE : {meilleur['MAE (MAD)']:.2f} MAD, avec {meilleur['Modèle']} et « {meilleur['Cohorte']} ».")

* **MAE** : l'erreur moyenne, dans l'unité du salaire.
* **RMSE** : pareil, mais les grosses erreurs comptent plus.
* **R²** : sur ce que le modèle prédit. Le salaire pour M1, M2 et M5 ; le Compa-Ratio pour M3, M4 et M6. Les deux ne se comparent pas entre eux.

Lecture : dans une colonne (un modèle), la ligne qui a la plus petite MAE est la meilleure façon de former les cohortes pour ce modèle. Si la dernière ligne, « Sans cohorte », a la plus petite MAE, apprendre par cohorte ne fait pas mieux que le modèle général.

Ce qui joue contre les cohortes précises : chaque modèle n'a que les salariés de sa cohorte pour apprendre. Plus la cohorte est précise, moins il en a. Un arbre ou un gradient boosting ont besoin de beaucoup de monde ; dans une petite cohorte, ils apprennent mal.

---
## 6. Le classement, du meilleur au pire

Pour chaque façon de former les cohortes, on garde son meilleur modèle : celui qui a la plus petite MAE. Puis on classe les façons de faire, de la plus petite MAE à la plus grande.

Le repère, « Sans cohorte », est classé avec les autres. Les façons de faire placées au-dessus de lui font mieux que le modèle général ; celles placées en dessous font moins bien.

In [ ]:
# 1. Pour chaque façon de former les cohortes (repère compris) : son meilleur modèle, celui qui a la plus petite MAE
classement = resultats.loc[resultats.groupby('Cohorte')['MAE (MAD)'].idxmin()]
# 2. Du meilleur au pire : de la plus petite MAE à la plus grande
classement = classement.sort_values('MAE (MAD)').reset_index(drop=True)
classement.index = classement.index + 1   # le rang : 1 pour le meilleur
classement.insert(2, 'Ce que fait le modèle', classement['Modèle'].map({nom: quoi for nom, (quoi, cible, modele) in MODELES.items()}))
display(classement.round(2))